# RustPower Lesson 2: Rapid Ingestion and Pandapower Compatibility 🏎️

Building grids manually is fine for simple systems; large networks call for
high-performance ingestion.

In this lesson we will:
1. Convert a **pandapower** network to RustPower in milliseconds.
2. Load standard **ZIP/CSV** case files.
3. Use the `Network` container for batch data handling — and see why it is
   **discarded** after ingestion (the ECS world is the single source of truth).

In [1]:
import rustpower
import pandapower.networks as nw
import pandapower as pp
import numpy as np
import time

print(f"RustPower Core Ready! Version: {rustpower.version()}")

RustPower Core Ready! Version: 0.5.0-rc.2

## 1. Direct conversion from pandapower

`PowerGrid.from_pandapower(net)` ingests a live pandapower net in one call,
using batch memory operations in Rust. After ingestion the pandapower data model
is dropped — all subsequent edits and queries work on the ECS components.

In [2]:
net = nw.case118()

start = time.time()
grid = rustpower.PowerGrid.from_pandapower(net)
end = time.time()

print(f"Converted IEEE 118 ({grid.n_bus} buses) in {(end-start)*1000:.2f} ms")

report = grid.solve()
print(f"Converged in {report.iterations} iterations. "
      f"Vm range: [{np.min(np.abs(grid.v)):.4f}, {np.max(np.abs(grid.v)):.4f}]")

Converted IEEE 118 (118 buses) in 14.26 ms

Converged in 3 iterations. Vm range: [0.9430, 1.0500]

## 2. Loading from ZIP/CSV

RustPower reads the pandapower CSV format directly. `load_csv_zip` produces a
`Network` container, and `load_network` ingests it into a grid.

`load_network` has **replace semantics**: it clears any previous grid entities,
re-ingests, and rebuilds. Re-loading is guaranteed to be identical to a fresh
construction — no residue, no double-counted injections.

In [3]:
# 1. Load the data structure from disk (path relative to examples/)
rust_net = rustpower.load_csv_zip('../cases/IEEE118/data.zip')
print(f"Loaded {len(rust_net.bus)} buses and {len(rust_net.line)} lines from ZIP.")

# 2. Ingest into a new PowerGrid
grid_zip = rustpower.PowerGrid()
grid_zip.load_network(rust_net)

report = grid_zip.solve()
print(f"Grid from ZIP converged in {report.iterations} iterations.")

# 3. Replace semantics: reloading gives a bit-identical fresh grid
v_first = grid_zip.v.copy()
grid_zip.load_network(rust_net)
grid_zip.solve()
print(f"Reload residue: {np.linalg.norm(v_first - grid_zip.v):.2e} (expect 0)")

Loaded 118 buses and 173 lines from ZIP.

Grid from ZIP converged in 3 iterations.

Reload residue: 0.00e+00 (expect 0)

## 3. Manipulating the `Network` container

`rustpower.Network` is a transparent **ingestion-boundary DTO** with *value
semantics*: `rust_net.trafo` returns a **copy** of the table, so mutating an
element in place does nothing until you assign the list back. (Contrast with
grid handles, which are *live proxies* into the ECS world — Lesson 1, §4.)

The pattern is read → modify → write back → ingest.

In [4]:
# Read: a copy of the transformer table
trafos = rust_net.trafo
t0 = trafos[0]
print(f"First Transformer: {t0.vn_hv_kv}/{t0.vn_lv_kv} kV, vk={t0.vk_percent}%")

# Modify the copy, then write the table back (value semantics!)
t0.vk_percent = 300.0
rust_net.trafo = trafos

# Ingest the modified network into a fresh grid
grid_modified = rustpower.PowerGrid()
grid_modified.load_network(rust_net)
report = grid_modified.solve()

dv = np.linalg.norm(grid_zip.v - grid_modified.v)
print(f"Voltage solution shift caused by the impedance change: |dv| = {dv:.2e}")

First Transformer: 345.0/138.0 kV, vk=264.33%

Voltage solution shift caused by the impedance change: |dv| = 2.31e-02